# R2-S1 - Protocol B mel statistics, crop bank and LR feature tables (Colab)

CPU runtime is enough (a GPU is not used). Run all cells top to bottom. It is safe to rerun after a disconnect: finished shards are kept
on Drive and skipped, and a finished build is detected and not repeated.

**What it produces** in `data/features/r2_s1_B/` on Drive (sign-off: K = 5, all valid frames of every B-train recording, train crops and validation chunks only):
1. `crop_bank_B_v1.parquet` - the fixed 5 crops per B-train recording (Usman's R2-U1 must use this same file);
2. `mel_stats_B_v1.npz` - per-bin log-mel mean and std over **every valid frame of every retained B-train recording**;
3. `lr_train_crops_B_v1.npz` - 67 handcrafted features for each crop;
4. `lr_validation_B_v1.npz` - 67 handcrafted features for every B validation chunk (all three roles);
5. `r2_s1_manifest.json` - SHA-256 of every output, the split/cohort/chunk/schema hashes, counts and checks;
6. `reports/r2/r2_s1_colab_<timestamp>.json` - the evidence report reviewed afterwards.

**What it never touches:** B test recordings (not read, not listed), Chatterbox (absent from B train and validation, and checked again), Protocol A.

Steps: check the synced code -> pre-flight on the locked inputs -> unit tests -> copy shards to local disk -> build -> independent cross-checks -> evidence.

In [ ]:
import os, sys, json, subprocess, time, hashlib, platform, shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get("ECHOPROOF_ROOT", "/content/drive/MyDrive/PAI&AIES Project"))
assert ROOT.exists(), f"project folder not found: {ROOT}"

SOURCES = {"stats.py": "src/echoproof/features/stats.py", "r2_s1.py": "src/echoproof/features/r2_s1.py",
           "acoustic.py": "src/echoproof/features/acoustic.py", "r2_s1_features.py": "scripts/r2_s1_features.py",
           "cohort.py": "src/echoproof/data/cohort.py", "splits.py": "src/echoproof/data/splits.py", "chunks.py": "src/echoproof/data/chunks.py",
           "datasets.py": "src/echoproof/training/datasets.py", "seeding.py": "src/echoproof/training/seeding.py",
           "contracts.py": "src/echoproof/contracts.py", "lr_detector.py": "src/echoproof/models/lr_detector.py",
           "test_r2_s1_features.py": "tests/test_r2_s1_features.py"}
missing = [k for k, rel in SOURCES.items() if not (ROOT / rel).exists()]
assert not missing, f"not found under {ROOT}: {missing} - sync these files to Drive and rerun"
SHA = {k: hashlib.sha256((ROOT / rel).read_bytes()).hexdigest() for k, rel in SOURCES.items()}
print(json.dumps({k: v[:12] for k, v in SHA.items()}, indent=2))

In [ ]:
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")   # spawned workers import echoproof too
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "src"))

import numpy, scipy, sklearn, librosa, pyarrow
ENV = {"python": platform.python_version(), "numpy": numpy.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__,
       "librosa": librosa.__version__, "pyarrow": pyarrow.__version__, "platform": platform.platform(), "cpu_count": os.cpu_count()}
print(json.dumps(ENV, indent=2))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

INPUTS = {"cohort": ROOT / "data/manifests/cohort_v1.parquet", "split": ROOT / "data/splits/protocol_B_v1.parquet",
          "chunks": ROOT / "data/processed/chunks_v1.parquet", "split_report": ROOT / "reports/r1/split_report.json"}
for name, path in INPUTS.items():
    assert path.exists(), f"missing locked input {name}: {path}"

# The GR1 freeze: the cohort manifest after the repair and the shard_14 rename. A different file means this Drive copy is not the locked one.
EXPECTED_COHORT_FILE_SHA256 = "51af27a2dd5c0577bf9c232287ac73b2f66af472ce3308721af83b2e48e8e816"
INPUT_SHA = {name: sha256_file(path) for name, path in INPUTS.items() if name != "split_report"}
print({k: v[:12] for k, v in INPUT_SHA.items()})
assert INPUT_SHA["cohort"] == EXPECTED_COHORT_FILE_SHA256, "the cohort manifest on Drive is not the frozen GR1 file - stop and compare it with the local copy"

import pyarrow.parquet as pq
FROZEN_B = json.loads(INPUTS["split_report"].read_text())["protocols"]["B"]["split_sha256"]
SPLIT_HASH_IN_FILE = {json.loads(r["payload_json"])["split_hash"] for r in pq.read_table(INPUTS["split"]).to_pylist()}
assert SPLIT_HASH_IN_FILE == {FROZEN_B}, f"protocol B split hash {SPLIT_HASH_IN_FILE} != frozen {FROZEN_B}"
print("protocol B split hash matches the frozen value:", FROZEN_B[:12])

from echoproof.features.acoustic import FEATURE_NAMES, FEATURE_SCHEMA_HASH
print("feature columns:", len(FEATURE_NAMES), "| schema hash:", FEATURE_SCHEMA_HASH[:12])

## 1. Unit tests on this runtime

In [ ]:
proc = subprocess.run([sys.executable, "-m", "pytest", "tests/test_r2_s1_features.py", "tests/test_r2_lr_detector.py", "-q", "-p", "no:cacheprovider"],
                      cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
print(proc.stdout[-3000:] + proc.stderr[-1500:])
UNIT_TESTS_PASSED = proc.returncode == 0
PYTEST_SUMMARY = next((l.strip() for l in reversed(proc.stdout.splitlines()) if " passed" in l or " failed" in l or " error" in l), "no summary line")
print("unit tests:", "PASSED" if UNIT_TESTS_PASSED else "FAILED", "|", PYTEST_SUMMARY)
assert UNIT_TESTS_PASSED, "do not build on a runtime where the tests fail"

## 2. Copy the shards from Drive to local disk (reading 15 tar files from Drive member by member is slow and fragile)

In [ ]:
SHARD_SRC = ROOT / "data/processed/cohort_v1"
SHARD_DIR = Path("/content/shards")
SHARD_DIR.mkdir(parents=True, exist_ok=True)
rows = pq.read_table(INPUTS["cohort"], columns=["canonical_path", "retained"]).to_pylist()
needed = sorted({r["canonical_path"].split("::")[0] for r in rows if r["retained"]})
print(len(needed), "shards referenced by the manifest:", needed)
t0 = time.perf_counter()
for name in needed:
    src, dst = SHARD_SRC / name, SHARD_DIR / name
    assert src.exists(), f"shard missing on Drive: {src}"
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        continue
    for attempt in range(4):
        try:
            tmp = dst.with_name(dst.name + ".part")
            shutil.copyfile(src, tmp)
            assert tmp.stat().st_size == src.stat().st_size, "size differs after copy"
            os.replace(tmp, dst)
            break
        except OSError as exc:
            print(f"  {name}: attempt {attempt + 1} failed ({exc}); retrying")
            time.sleep(5 * 2 ** attempt)
    else:
        raise RuntimeError(f"could not copy {name}")
    print(f"  copied {name} ({dst.stat().st_size / 1e6:.0f} MB)")
COPY_SECONDS = round(time.perf_counter() - t0, 1)
print("shards ready in", SHARD_DIR, "|", COPY_SECONDS, "s")

## 3. Build (about 15-30 minutes on 2 CPUs; progress is printed per shard)

In [ ]:
from echoproof.features import r2_s1

OUT_DIR = ROOT / "data/features/r2_s1_B"
BUILD_SECONDS = None
if (OUT_DIR / r2_s1.MANIFEST_NAME).exists():
    print("already built - loading the existing manifest")
    MANIFEST = json.loads((OUT_DIR / r2_s1.MANIFEST_NAME).read_text())
else:
    t0 = time.perf_counter()
    MANIFEST = r2_s1.build(ROOT, SHARD_DIR, OUT_DIR, k=5, seed=42, workers=os.cpu_count() or 1, resume=True, progress=print)
    BUILD_SECONDS = round(time.perf_counter() - t0, 1)
print(json.dumps({k: MANIFEST[k] for k in ("plan_sha256", "counts", "mel_stats", "seconds_total")}, indent=2))

## 4. Independent cross-checks of the files on Drive

In [ ]:
import numpy as np
from collections import Counter
from echoproof.features.stats import load_mel_stats

problems = []

# 4a. every output file still has the hash the manifest recorded
OUTPUT_HASHES = {}
for key, info in MANIFEST["outputs"].items():
    actual = sha256_file(OUT_DIR / info["file"])
    OUTPUT_HASHES[key] = actual
    if actual != info["sha256"]:
        problems.append(f"{key}: file hash differs from the manifest")

# 4b. counts re-derived straight from the chunk manifest, not from the build's own plan
chunk_rows = [json.loads(r["payload_json"]) for r in pq.read_table(INPUTS["chunks"]).to_pylist()]
b_validation = [c for c in chunk_rows if c["protocol_id"] == "protocol:B" and c["partition"] == "validation"]
b_test_chunks = sum(1 for c in chunk_rows if c["protocol_id"] == "protocol:B" and c["partition"] == "test")
train = np.load(OUT_DIR / r2_s1.OUTPUT_NAMES["train_features"], allow_pickle=False)
val = np.load(OUT_DIR / r2_s1.OUTPUT_NAMES["validation_features"], allow_pickle=False)
stats = load_mel_stats(OUT_DIR / r2_s1.OUTPUT_NAMES["mel_stats"])
if len(b_validation) != len(val["y"]):
    problems.append(f"validation rows {len(val['y'])} != B validation chunks in the chunk manifest {len(b_validation)}")
if set(val["chunk_id"].tolist()) != {c["chunk_id"] for c in b_validation}:
    problems.append("validation chunk ids differ from the chunk manifest")

# 4c. scope: no test rows, no Chatterbox, speakers disjoint, both classes
test_ids = {json.loads(r["payload_json"])["recording_id"] for r in pq.read_table(INPUTS["split"]).to_pylist()
            if json.loads(r["payload_json"])["partition"] == "test"}
used = set(train["recording_id"].tolist()) | set(val["recording_id"].tolist())
if used & test_ids:
    problems.append("test recordings appear in the outputs")
if any(":chatterbox:" in r for r in used):
    problems.append("Chatterbox recordings appear in the outputs")
if set(train["group_id"].tolist()) & set(val["group_id"].tolist()):
    problems.append("a speaker appears in train and validation")
if set(train["y"].tolist()) != {0, 1} or set(val["y"].tolist()) != {0, 1}:
    problems.append("a class is missing")

# 4d. feature health
nan_by_column = {FEATURE_NAMES[j]: int(np.isnan(np.concatenate([train["X"], val["X"]])[:, j]).sum())
                 for j in range(len(FEATURE_NAMES)) if np.isnan(np.concatenate([train["X"], val["X"]])[:, j]).any()}
unexpected = sorted(set(nan_by_column) - r2_s1.EXPECTED_NAN_COLUMNS)
if unexpected:
    problems.append(f"NaN in unexpected columns: {unexpected}")
if not (train["X"].shape[1] == len(FEATURE_NAMES) == val["X"].shape[1]):
    problems.append("feature width differs from the schema")
if not np.isfinite(np.nan_to_num(np.concatenate([train["X"], val["X"]]), nan=0.0)).all():
    problems.append("infinite feature values")

SUMMARY = {
    "train_recordings": len(set(train["recording_id"].tolist())), "train_crops": int(len(train["y"])),
    "train_crops_by_component": dict(Counter(train["component"].tolist())), "train_label_counts": dict(Counter(train["y"].tolist())),
    "validation_recordings": len(set(val["recording_id"].tolist())), "validation_chunks": int(len(val["y"])),
    "validation_chunks_by_role": dict(Counter(val["role"].tolist())), "validation_chunks_by_component": dict(Counter(val["component"].tolist())),
    "mel_frames": stats["frames"], "mel_recordings": stats["recordings"],
    "mel_std_range": [float(stats["std"].min()), float(stats["std"].max())],
    "b_test_chunks_in_manifest_not_used": b_test_chunks, "nan_by_column": nan_by_column,
}
print(json.dumps(SUMMARY, indent=2))
print("problems:", problems or "none")

## 5. Verdict and evidence

In [ ]:
build = MANIFEST
checks = {
    "unit_tests_passed": UNIT_TESTS_PASSED,
    "build_verification_passed": build["verification"]["passed"],
    "spot_check_mismatches": build["spot_check"]["mismatches"],
    "streaming_vs_two_pass_max_abs": build["spot_check"]["streaming_vs_two_pass_max_abs"],
    "cohort_manifest_is_frozen_file": INPUT_SHA["cohort"] == EXPECTED_COHORT_FILE_SHA256,
    "split_hash_is_frozen_value": build["inputs"]["split_sha256"] == FROZEN_B == build["inputs"]["split_sha256_frozen_in_report"],
    "feature_schema_hash_matches": build["inputs"]["feature_schema_hash"] == FEATURE_SCHEMA_HASH,
    "cross_check_problems": problems,
}
verified = (checks["unit_tests_passed"] and checks["build_verification_passed"] and not checks["spot_check_mismatches"]
            and checks["streaming_vs_two_pass_max_abs"] < 1e-9 and checks["cohort_manifest_is_frozen_file"]
            and checks["split_hash_is_frozen_value"] and checks["feature_schema_hash_matches"] and not problems)
evidence = {"schema_version": 1, "task": "R2-S1", "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "environment": ENV, "source_sha256": SHA, "input_sha256": INPUT_SHA, "output_sha256": OUTPUT_HASHES,
            "verdict": {"verified": bool(verified), "pytest_summary": PYTEST_SUMMARY, **checks},
            "summary": SUMMARY, "build_manifest": build, "timing": {"shard_copy_seconds": COPY_SECONDS, "build_seconds_this_session": BUILD_SECONDS,
                                                                  "build_seconds_recorded": build["seconds_total"]}}
out_dir = ROOT / "reports" / "r2"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / f"r2_s1_colab_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.json"
path.write_text(json.dumps(evidence, indent=2, sort_keys=True, default=str), encoding="utf-8")
print(json.dumps(evidence["verdict"], indent=2, default=str))
print("\nevidence written to:", path)